A gente define o objetivo, se eh replicar o indice ou montar uma carteira equiponderada que perfomane melhor que o indice

## Imports

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pathlib
from scipy import stats
import re
import matplotlib.lines as mlines
import warnings
from IPython.display import display
from pathlib import Path

warnings.filterwarnings("ignore")

### Global Variables

In [2]:
BASE_DIR = pathlib.Path().resolve().parent
BRONZE_DIR = BASE_DIR / 'data' / 'bronze'
SILVER_DIR = BASE_DIR / 'data' / 'silver'
GOLD_DIR = BASE_DIR / 'data' / 'gold'

### Functions

In [3]:
def count_outliers_iqr(series):
    """Robust method based on quartiles"""
    if series.empty or not np.issubdtype(series.dtype, np.number): 
        return 0
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return ((series < (q1 - 1.5 * iqr)) | (series > (q3 + 1.5 * iqr))).sum()

def count_outliers_zscore(series, threshold=3):
    """Method based on standard deviation (Z-score)"""
    if series.empty or len(series) < 2 or not np.issubdtype(series.dtype, np.number): 
        return 0
    z_scores = np.abs(stats.zscore(series, nan_policy='omit'))
    return (z_scores > threshold).sum()

def is_valid_isin(isin):
    """Validates if the ISIN follows the international format"""
    if pd.isna(isin): 
        return False
    return bool(re.match(r"^[A-Z]{2}[A-Z0-9]{9}\d{1}$", str(isin)))

### Plots

In [4]:
def plot_full_diagnostic(asset_id, data_dict, quality_df, metadata_df):
    """
    Plots the NAV time series with all identified inconsistencies 
    and displays a summary quality table.
    """
    # 1. Data Retrieval
    try:
        # Match ID type to the index of quality_df (usually int or str)
        target_id = type(quality_df.index[0])(asset_id)
        asset_quality = quality_df.loc[[target_id]]
        
        source_key = asset_quality['source_key'].values[0]
        fund_name = metadata_df.loc[target_id]['name']
        df = data_dict[source_key].copy()
    except Exception as e:
        print(f"Error: Asset ID {asset_id} not found in the dataset. {e}")
        return

    # --- 2. Identifying Plot Markers ---
    nav = df['nav']
    
    # Outliers: IQR Method
    q1, q3 = nav.quantile(0.25), nav.quantile(0.75)
    iqr = q3 - q1
    iqr_outliers = df[(nav < (q1 - 1.5 * iqr)) | (nav > (q3 + 1.5 * iqr))]
    
    # Outliers: Z-Score Method (> 3 standard deviations)
    z_scores = np.abs(stats.zscore(nav, nan_policy='omit'))
    z_outliers = df[z_scores > 3]
    
    # Extreme Volatility (> 15% daily change)
    extreme_vol = df[nav.pct_change().abs() > 0.15]
    
    # Invalid Values (<= 0)
    invalid_vals = df[nav <= 0]

    # --- 3. Visualization ---
    plt.figure(figsize=(16, 7))
    
    # Handle Gaps for Visualization (Reindex to daily frequency)
    full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='D')
    df_daily = df.reindex(full_range)
    df_interp = df_daily['nav'].interpolate(method='linear')
    
    # Plotting Lines
    plt.plot(df_interp.index, df_interp, color='red', linestyle='--', alpha=0.3, label='Gap Jump')
    plt.plot(df_daily.index, df_daily['nav'], color='royalblue', lw=2, label='Actual NAV', zorder=2)

    # Scatter Plotting Inconsistencies
    if not iqr_outliers.empty:
        plt.scatter(iqr_outliers.index, iqr_outliers['nav'], color='orange', 
                    label='IQR Outlier', s=50, alpha=0.7, zorder=3)
    
    if not z_outliers.empty:
        plt.scatter(z_outliers.index, z_outliers['nav'], color='red', marker='x', 
                    label='Z-Score Outlier (>3σ)', s=80, zorder=4)
        
    if not extreme_vol.empty:
        plt.scatter(extreme_vol.index, extreme_vol['nav'], edgecolors='black', 
                    facecolors='none', s=120, lw=1.5, label='Extreme Volatility (>15%)', zorder=5)

    if not invalid_vals.empty:
        plt.scatter(invalid_vals.index, invalid_vals['nav'], color='black', marker='s', 
                    label='Invalid Value (<=0)', s=100, zorder=6)

    # Formatting the Plot
    plt.title(f"Full Diagnostic: {fund_name} (ID: {asset_id})", fontsize=15, fontweight='bold', pad=20)
    plt.ylabel("NAV Value")
    plt.xlabel("Date")
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10)
    plt.tight_layout()
    plt.show()

    # --- 4. Summary Table Display ---
    display(Markdown(f"### Quality Summary Report: Asset {asset_id}"))
    
    # Select specific columns to show in the summary
    summary_cols = [
        'total_rows', 'gaps_data_%', 'outliers_zscore_%', 
        'stale_prices_%', 'invalid_values_%', 'extreme_volatility_%'
    ]
    
    # Transpose for better vertical readability
    summary_table = asset_quality[summary_cols].T
    summary_table.columns = ['Current Value']
    
    # Add a 'Status' column based on thresholds (optional logic)
    display(summary_table.style.format("{:.2f}").background_gradient(cmap='YlOrRd', axis=0))

### Generate DFs

In [5]:
def process_fund_data(asset_dict, export_path=None):
    """
    Processes a dictionary of fund DataFrames to extract quality metrics, 
    metadata, and identify duplicates.
    
    Args:
        asset_dict (dict): Dictionary where keys are IDs and values are DataFrames.
        export_path (str/Path, optional): Directory to save the resulting CSVs.
        
    Returns:
        tuple: (df_quality, df_metadata, df_duplicated)
    """
    quality_list = []
    metadata_list = []
    all_duplicates_list = []

    print(f"Starting processing for {len(asset_dict)} assets...")

    for key, df in asset_dict.items():
        total_rows = len(df)
        if total_rows == 0:
            continue

        # 1. Data Standardization
        if not isinstance(df.index, pd.DatetimeIndex):
            df.index = pd.to_datetime(df.index)

        # 2. Metadata Extraction
        af_id = df['allfunds_id'].iloc[0] if 'allfunds_id' in df.columns else "N/A"
        isin = df['isin'].iloc[0] if 'isin' in df.columns else "N/A"
        name = df['name'].iloc[0] if 'name' in df.columns else "N/A"
        
        metadata_list.append({'allfunds_id': af_id, 'isin': isin, 'name': name})

        # 3. Quality Diagnosis
        nav_series = df['nav']
        diffs = nav_series.diff()
        returns = nav_series.pct_change().abs()
        
        count_iqr = count_outliers_iqr(nav_series)
        count_zscore = count_outliers_zscore(nav_series)
        
        expected_days = (df.index.max() - df.index.min()).days + 1
        count_gaps = max(0, expected_days - total_rows)
        
        count_stale = (diffs == 0).sum()
        count_invalid = (nav_series <= 0).sum()
        count_volat = (returns > 0.15).sum()

        quality_list.append({
            'source_key': key,
            'allfunds_id': af_id,
            'nav_nulls': nav_series.isnull().sum(),
            'first_index': df.index[0],
            'last_index': df.index[-1],
            'index_is_unique': df.index.is_unique,
            'index_duplicated_sum': df.index.duplicated().sum(),
            'total_rows': total_rows,
            'outliers_iqr_abs': count_iqr,
            'outliers_iqr_%': round((count_iqr / total_rows) * 100, 2),
            'outliers_zscore_abs': count_zscore,
            'outliers_zscore_%': round((count_zscore / total_rows) * 100, 2),
            'gaps_data_%': round((count_gaps / expected_days) * 100, 2) if expected_days > 0 else 0,
            'stale_prices_%': round((count_stale / total_rows) * 100, 2),
            'invalid_values_%': round((count_invalid / total_rows) * 100, 2),
            'extreme_volatility_%': round((count_volat / total_rows) * 100, 2),
            'is_isin_valid': is_valid_isin(isin)
        })

        # 4. Duplicates Capture
        if df.index.duplicated().any():
            duplicates = df[df.index.duplicated(keep=False)].copy()
            duplicates['source_key'] = key
            all_duplicates_list.append(duplicates)

        # 5. In-place Cleaning (Modifies the input dictionary)
        asset_dict[key] = df.drop(columns=['isin', 'name'], errors='ignore')

    # --- CONSOLIDATION ---
    df_quality = pd.DataFrame(quality_list).set_index('allfunds_id')
    df_metadata = pd.DataFrame(metadata_list).drop_duplicates(subset=['allfunds_id']).set_index('allfunds_id')
    df_duplicated = pd.concat(all_duplicates_list) if all_duplicates_list else pd.DataFrame()

    # --- OPTIONAL EXPORT ---
    if export_path:
        path = Path(export_path)
        path.mkdir(parents=True, exist_ok=True)
        df_quality.to_csv(path / 'quality_data.csv')
        df_metadata.to_csv(path / 'metadata.csv')
        if not df_duplicated.empty:
            df_duplicated.to_csv(path / 'duplicated_rows.csv')
        print(f"Files successfully saved to: {export_path}")

    return df_quality, df_metadata, df_duplicated


### Filtering Data

In [6]:
def filter_assets_by_quality(df_quality, upper_q=0.80, lower_q=0.20, date_q=0.95, export_path=None):
    """
    Filters assets based on statistical quality thresholds calculated from quantiles.
    
    Args:
        df_quality (pd.DataFrame): The quality metrics dataframe.
        upper_q (float): Quantile for error metrics (max allowed). Defaults to 0.80.
        lower_q (float): Quantile for history metrics (min required). Defaults to 0.20.
        date_q (float): Quantile for the first_index (cutoff). Defaults to 0.95.
        export_path (str/Path, optional): Directory or file path to save the filtered CSV.
        
    Returns:
        pd.DataFrame: The filtered dataframe containing high-quality assets.
    """
    
    # 1. Dynamic Threshold Calculation
    # We want to exclude the worst performers (top X% of errors)
    thresholds = {
        'min_rows': df_quality['total_rows'].quantile(lower_q),
        'cutoff_date': df_quality['first_index'].quantile(date_q),
        'max_gaps': df_quality['gaps_data_%'].quantile(upper_q),
        'max_outliers': df_quality['outliers_zscore_%'].quantile(upper_q),
        'max_stale': df_quality['stale_prices_%'].quantile(upper_q),
        'max_volatility': df_quality['extreme_volatility_%'].quantile(upper_q)
    }

    print("--- Calculated Statistical Thresholds ---")
    for metric, value in thresholds.items():
        print(f"{metric}: {value}")

    # 2. Applying Filter Criteria
    # Using a list of conditions for better readability
    conditions = (
        (df_quality['index_is_unique'] == True) &
        (df_quality['index_duplicated_sum'] == 0) &
        (df_quality['total_rows'] >= thresholds['min_rows']) &
        (df_quality['first_index'] <= thresholds['cutoff_date']) &
        (df_quality['outliers_zscore_%'] <= thresholds['max_outliers']) &
        (df_quality['gaps_data_%'] <= thresholds['max_gaps']) &
        (df_quality['stale_prices_%'] <= thresholds['max_stale']) &
        (df_quality['invalid_values_%'] == 0) &
        (df_quality['extreme_volatility_%'] <= thresholds['max_volatility'])
    )

    df_filtered = df_quality.loc[conditions].copy()

    # 3. Summary Results
    total = len(df_quality)
    approved = len(df_filtered)
    pct = round((approved / total) * 100, 2) if total > 0 else 0
    
    print(f"\nFiltering Result: {approved} / {total} assets approved ({pct}%).")

    # 4. Export logic
    if export_path:
        save_path = Path(export_path)
        # If export_path is a directory, add filename. If it's a file, use as is.
        if save_path.is_dir() or not save_path.suffix:
            save_path.mkdir(parents=True, exist_ok=True)
            save_path = save_path / 'quality_data_filtered.csv'
            
        df_filtered.to_csv(save_path)
        print(f"Filtered report saved to: {save_path}")

    return df_filtered

### Main

In [7]:
def main():
    # 1. Load the raw data
    pickle_path = BRONZE_DIR / 'navs.pickle'
    
    if not pickle_path.exists():
        print(f"Error: {pickle_path} not found.")
        return

    raw_data = pd.read_pickle(pickle_path)
    asset_ids = list(raw_data.keys())
    
    print(f"Loaded {len(asset_ids)} assets from Bronze layer.")

    # 2. Run Processing (The function we created earlier)
    df_quality, df_metadata, df_duplicated = process_fund_data(
        raw_data, 
        export_path=SILVER_DIR
    )

    # 3. Run Filtering (The second function we created)
    df_filtered = filter_assets_by_quality(
        df_quality, 
        upper_q=0.80, 
        lower_q=0.20, 
        export_path=SILVER_DIR
    )

    return raw_data, df_metadata, df_quality, df_filtered

# --- EXECUTION ---
if __name__ == "__main__":
    data, metadata, quality, filtered = main()

Loaded 24822 assets from Bronze layer.
Starting processing for 24822 assets...
Files successfully saved to: C:\Users\piett\OneDrive\Desktop\Pietro\Master MIAX\Clases\3.Inteligencia Artificial Basica\Taller\Supervisado\data\silver
--- Calculated Statistical Thresholds ---
min_rows: 898.0
cutoff_date: 2019-06-14 00:00:00
max_gaps: 33.41
max_outliers: 0.4
max_stale: 5.22
max_volatility: 0.0

Filtering Result: 10317 / 24822 assets approved (41.56%).
Filtered report saved to: C:\Users\piett\OneDrive\Desktop\Pietro\Master MIAX\Clases\3.Inteligencia Artificial Basica\Taller\Supervisado\data\silver\quality_data_filtered.csv
